# RAG Document Ingestion with Ray Data

This notebook ingests PDF documents into a [Milvus](https://milvus.io/) vector
database for Retrieval-Augmented Generation (RAG). It uses:

- **[Ray Data](https://docs.ray.io/en/latest/data/data.html)** for distributed processing
- **[Docling](https://github.com/DS4SD/docling)** for PDF parsing and chunking
- **[sentence-transformers](https://www.sbert.net/)** for embedding generation (CPU)
- **[Milvus](https://milvus.io/)** for vector storage

## Architecture

![RAG Pipeline Architecture](./assets/rag-architecture.png)

The pipeline runs as a **RayJob** on an existing **RayCluster**. Ray Data
**streams** the three stages — workers overlap (Docling is usually the slowest).
The pipeline code lives in `docling_milvus_process.py`. Step 4 walks through
each section so you can understand what runs on the cluster before submitting.

## Prerequisites

- **Red Hat OpenShift AI** (RHOAI) with the KubeRay operator
- An **existing RayCluster** (see `manifests/raycluster-rag-optimized.yaml`)
- A **ReadWriteMany PVC** mounted on all workers with input PDFs
- **Milvus** accessible from the Ray cluster
- **CodeFlare SDK** installed in your workbench (`pip install codeflare-sdk`)


In [9]:
!pip install codeflare-sdk

---
## Step 1 -- Import Dependencies


In [11]:
import os

from codeflare_sdk import RayJob

---
## Step 2 -- Connect to OpenShift

Replace the token and server URL below. You can get a login token from the
OpenShift web console under **Copy login command**.


In [12]:
# ---- REPLACE THESE WITH YOUR VALUES ----
TOKEN = "sha256~2gnPsjk7o7TehxWjTtR3yT4FfeF2wqcEyi1jaES0w4w"
API_URL = "https://api.bkeane.ngxc.p3.openshiftapps.com:443"

In [13]:
!oc login --token={TOKEN} --server={API_URL}

Logged into "https://api.bkeane.ngxc.p3.openshiftapps.com:443" as "htpasswd-cluster-admin-user" using the token provided.

You have access to 101 projects, the list has been suppressed. You can list all projects with 'oc projects'

Using project "rag-example".


---
## Step 3 -- Configure Pipeline Parameters

All tunable parameters are defined here. The defaults below are optimized for
throughput on an 8-worker CPU cluster (32 CPUs).

Embedding uses **sentence-transformers** (`all-MiniLM-L6-v2` by default, 384 dimensions).

**Why it looks like only Docling is running:** Ray Data runs *parse → embed → Milvus*
in order. Docling is usually the slowest stage, so workers are busy parsing for most
of the job. Embedding and Milvus appear later (or overlap a little via pipelining).
With `NUM_EMBED_ACTORS=1`, embedding is a **single** actor, which exaggerates that
effect. Raise `NUM_EMBED_ACTORS` (default below is 4) to parallelize embeddings.


In [16]:
# Cluster
EXISTING_CLUSTER_NAME = "raytest"
NAMESPACE = "rag-example"

# Storage (PVC mounted on all Ray workers)
PVC_MOUNT_PATH = "/mnt/data"
INPUT_PATH = "input/arxiv"

# Milvus
MILVUS_HOST = "milvus-milvus.milvus.svc.cluster.local"
MILVUS_PORT = "19530"
MILVUS_DB = "default"
MILVUS_COLLECTION = "rag_documents"

# Embedding (sentence-transformers, CPU)
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIM = "384"

# Pipeline tuning
NUM_FILES = "0"  # PDFs to process (0 = all)
NUM_ACTORS = "6"  # Docling parsing actors
CPUS_PER_ACTOR = "4"  # CPUs per Docling actor
NUM_EMBED_ACTORS = (
    "4"  # Parallel embedding actors
)
NUM_MILVUS_ACTORS = "2"  # Milvus write actors (I/O-bound; server is the limit)
BATCH_SIZE = "1"  # PDFs per actor batch
EMBED_MAP_BATCH_SIZE = "32"
MILVUS_BATCH_SIZE = "64"
REPARTITION_FACTOR = "10"
CHUNK_MAX_TOKENS = "128"  # Max tokens per chunk (<=512 for BERT)

print(f"Cluster:    {EXISTING_CLUSTER_NAME} in {NAMESPACE}")
print(f"Input:      {PVC_MOUNT_PATH}/{INPUT_PATH}")
print(f"Milvus:     {MILVUS_HOST}:{MILVUS_PORT}/{MILVUS_DB}.{MILVUS_COLLECTION}")
print(f"Embedding:  {EMBEDDING_MODEL}")
print(
    f"Actors:     {NUM_ACTORS} docling x {CPUS_PER_ACTOR} CPUs, {NUM_EMBED_ACTORS} embed, {NUM_MILVUS_ACTORS} milvus"
)
print(f"Files:      {'ALL' if NUM_FILES == '0' else NUM_FILES}")

Cluster:    raytest in rag-example
Input:      /mnt/data/input/arxiv
Milvus:     milvus-milvus.milvus.svc.cluster.local:19530/default.rag_documents
Embedding:  sentence-transformers/all-MiniLM-L6-v2
Actors:     6 docling x 4 CPUs, 4 embed, 2 milvus
Files:      ALL


---
## Step 4 -- Pipeline Code Walkthrough

The pipeline code lives in `docling_milvus_process.py` and runs remotely on
the RayCluster (not in this notebook). The following sections walk through
each part so you can understand what happens when the RayJob executes.


### Parameters

All configuration is read from environment variables. The notebook sets these
via the RayJob's `env_vars` dict (Step 6). Defaults match the values in Step 3.

```python
NUM_ACTORS = int(os.environ.get("NUM_ACTORS", "6"))
NUM_EMBED_ACTORS = int(os.environ.get("NUM_EMBED_ACTORS", "4"))
NUM_MILVUS_ACTORS = int(os.environ.get("NUM_MILVUS_ACTORS", "2"))
CPUS_PER_ACTOR = int(os.environ.get("CPUS_PER_ACTOR", "4"))

BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "1"))
EMBED_MAP_BATCH_SIZE = int(os.environ.get("EMBED_MAP_BATCH_SIZE", "32"))

PVC_MOUNT_PATH = os.environ.get("PVC_MOUNT_PATH", "/mnt/data")
INPUT_PATH = os.environ.get("INPUT_PATH", "input/pdfs")
NUM_FILES = int(os.environ.get("NUM_FILES", "0"))

MILVUS_HOST = os.environ.get("MILVUS_HOST", "milvus-milvus.milvus.svc.cluster.local")
MILVUS_PORT = int(os.environ.get("MILVUS_PORT", "19530"))
COLLECTION_NAME = os.environ.get("MILVUS_COLLECTION", "rag_documents")

EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
EMBEDDING_DIM = int(os.environ.get("EMBEDDING_DIM", "384"))
CHUNK_MAX_TOKENS = int(os.environ.get("CHUNK_MAX_TOKENS", "128"))
```


### Helpers

`_collect_pdf_paths` walks the PVC directory to find PDFs (up to `NUM_FILES`).
`_configure_ray_context` tunes Ray Data for throughput: tolerates up to 50
errored blocks, sets a 2 MB target block size, and enables progress bars.

```python
def _collect_pdf_paths(input_path: str, limit: int) -> List[str]:
    root = Path(input_path)
    if not root.is_dir():
        return []
    out = []
    for p in root.rglob("*.pdf"):
        if p.is_file():
            out.append(str(p))
            if limit > 0 and len(out) >= limit:
                break
    return out


def _configure_ray_context():
    ctx = ray.data.DataContext.get_current()
    ctx.max_errored_blocks = 50
    if hasattr(ctx, "target_max_block_size"):
        ctx.target_max_block_size = 2 * 1024 * 1024
    if hasattr(ctx, "enable_rich_progress_bars"):
        ctx.enable_rich_progress_bars = True
```


### Stage 1: DoclingChunkActor

A Ray Data stateful actor (class-based UDF) that parses PDFs and splits them
into token-bounded chunks. Each actor in the pool initializes a Docling
`DocumentConverter` and a `HybridChunker` once in `__init__`, then reuses
them across batches.

For each PDF in the batch:
1. Read raw bytes into a `DocumentStream`
2. Convert to a structured Docling document (no OCR, CPU only)
3. Split into chunks of up to `CHUNK_MAX_TOKENS` tokens

Files that fail to parse or are too small are skipped with a log message.

```python
class DoclingChunkActor:
    def __init__(self):
        from docling.chunking import HybridChunker
        from docling.document_converter import DocumentConverter, PdfFormatOption

        pipeline_options = PdfPipelineOptions()
        pipeline_options.do_ocr = False
        pipeline_options.do_table_structure = True
        pipeline_options.accelerator_options = AcceleratorOptions(
            num_threads=CPUS_PER_ACTOR, device="cpu"
        )

        self.converter = DocumentConverter(...)
        self.chunker = HybridChunker(
            tokenizer=EMBEDDING_MODEL, max_tokens=CHUNK_MAX_TOKENS
        )

    def __call__(self, batch):
        for file_bytes, file_path in zip(batch["bytes"], batch["path"]):
            doc = self.converter.convert(DocumentStream(...)).document
            for chunk in self.chunker.chunk(doc):
                # emit: text, source_file, chunk_index, chunk_size_chars, num_pages
```


### Stage 2: SentenceTransformersEmbeddingActor

Loads a SentenceTransformer model once in `__init__` and encodes each batch
of text chunks into dense vectors. Uses `normalize_embeddings=True` so the
vectors work with the COSINE metric in Milvus.

```python
class SentenceTransformersEmbeddingActor:
    def __init__(self):
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer(EMBEDDING_MODEL)

    def __call__(self, batch):
        texts = list(batch["text"])
        embeddings = self.model.encode(
            texts, normalize_embeddings=True, show_progress_bar=False
        ).tolist()
        return {**batch, "embedding": embeddings}
```


### Stage 3: MilvusWriteActor

Opens a `MilvusClient` connection in `__init__` and inserts each batch of
embeddings in sub-batches of `MILVUS_BATCH_SIZE`. Text is truncated to
`MILVUS_TEXT_MAX_CHARS` (8192) to stay within the VARCHAR field limit.

```python
class MilvusWriteActor:
    def __init__(self):
        from pymilvus import MilvusClient
        self.milvus = MilvusClient(
            uri=f"http://{MILVUS_HOST}:{MILVUS_PORT}", db_name=MILVUS_DB
        )

    def __call__(self, batch):
        for i in range(0, len(texts), MILVUS_BATCH_SIZE):
            data = [{"source_file": ..., "text": ..., "embedding": ...}, ...]
            self.milvus.insert(collection_name=COLLECTION_NAME, data=data)
```


### Milvus Collection Setup

`setup_milvus_collection()` creates (or recreates) the target collection with
these fields:

| Field | Type | Notes |
|-------|------|-------|
| `id` | INT64 | Auto-generated primary key |
| `source_file` | VARCHAR(512) | Original PDF filename |
| `chunk_index` | INT64 | Chunk position within the document |
| `text` | VARCHAR(8192) | The chunk text |
| `embedding` | FLOAT_VECTOR | Dense vector (384 dims by default) |

An `IVF_FLAT` index with `COSINE` metric is built on the embedding field.

**Note:** The collection is dropped and recreated on every run for
reproducibility. For incremental ingestion, skip the drop or use a
different collection name.


### Pipeline Orchestration

The `_run_pipeline` function chains three `map_batches` stages using Ray Data
streaming execution. Each stage uses the `concurrency` parameter (recommended
over the deprecated `ActorPoolStrategy`) to set its actor pool size:

```python
# Stage 1: Parse + chunk (CPU-heavy, bottleneck — gets most resources)
ds = ds.map_batches(
    DoclingChunkActor,
    concurrency=NUM_ACTORS,        # 6 actors
    batch_size=BATCH_SIZE,
    num_cpus=CPUS_PER_ACTOR,       # 4 CPUs each = 24 CPUs
)

# Stage 2: Embed (CPU, parallel)
ds = ds.map_batches(
    SentenceTransformersEmbeddingActor,
    concurrency=NUM_EMBED_ACTORS,  # 4 actors x 1 CPU = 4 CPUs
    batch_size=EMBED_MAP_BATCH_SIZE,
    num_cpus=1,
)

# Stage 3: Write to Milvus (I/O-bound — fewer actors, server is the limit)
results = ds.map_batches(
    MilvusWriteActor,
    concurrency=NUM_MILVUS_ACTORS, # 2 actors x 1 CPU = 2 CPUs
    batch_size=MILVUS_BATCH_SIZE,
    num_cpus=1,
)
```

**Resource budget:** 6×4 + 4×1 + 2×1 = **30 CPUs** of 32 available, leaving
headroom for the driver. Ray's streaming executor overlaps stages: downstream
actors start as soon as upstream blocks are ready.

Results are consumed via `iter_batches` to stream metrics without holding the
full dataset in memory. The `run()` entrypoint reads PDFs from the PVC,
repartitions for parallelism, sets up the Milvus collection, runs the
pipeline, and prints a performance report with throughput stats.


---
## Step 5 -- Verify Prerequisites

Check that the RayCluster is running and the pipeline script exists.


In [15]:
# Verify the RayCluster is running
!oc get raycluster {EXISTING_CLUSTER_NAME} -n {NAMESPACE}

# Verify the processing script exists
script = os.path.join(os.getcwd(), "docling_milvus_process.py")
assert os.path.exists(script), f"Script not found: {script}"
print(f"Processing script: {script} ({os.path.getsize(script):,} bytes)")

NAME      DESIRED WORKERS   AVAILABLE WORKERS   CPUS   MEMORY   GPUS   STATUS   AGE
raytest   8                 8                   33     134Gi    0      ready    10d
Processing script: /opt/app-root/src/red-hat-ai-examples/examples/ray/data/rag/docling_milvus_process.py (29,498 bytes)


---
## Step 6 -- Submit the RayJob

This creates a RayJob CR that submits the processing script to the existing
RayCluster. The script reads all configuration from environment variables.
Docling, sentence-transformers, Logfire, and other dependencies are installed via
the RayJob `runtime_env` pip list. Set `LOGFIRE_TOKEN` in your shell (or Jupyter env)
before this cell so workers can send traces; otherwise replace the placeholder in `pipeline_env`.


In [17]:
pip_deps = [
    "opencv-python-headless",
    "pypdfium2",
    "pymilvus>=2.4.0",
    "sentence-transformers>=2.2.0",
    "logfire",
]

pipeline_env = {
    "PVC_MOUNT_PATH": PVC_MOUNT_PATH,
    "INPUT_PATH": INPUT_PATH,
    "MILVUS_HOST": MILVUS_HOST,
    "MILVUS_PORT": MILVUS_PORT,
    "MILVUS_DB": MILVUS_DB,
    "MILVUS_COLLECTION": MILVUS_COLLECTION,
    "EMBEDDING_MODEL": EMBEDDING_MODEL,
    "EMBEDDING_DIM": EMBEDDING_DIM,
    "NUM_FILES": NUM_FILES,
    "NUM_ACTORS": NUM_ACTORS,
    "CPUS_PER_ACTOR": CPUS_PER_ACTOR,
    "NUM_EMBED_ACTORS": NUM_EMBED_ACTORS,
    "NUM_MILVUS_ACTORS": NUM_MILVUS_ACTORS,
    "BATCH_SIZE": BATCH_SIZE,
    "EMBED_MAP_BATCH_SIZE": EMBED_MAP_BATCH_SIZE,
    "MILVUS_BATCH_SIZE": MILVUS_BATCH_SIZE,
    "REPARTITION_FACTOR": REPARTITION_FACTOR,
    "CHUNK_MAX_TOKENS": CHUNK_MAX_TOKENS,
    "HF_HOME": "/tmp/huggingface",
    "XDG_CACHE_HOME": "/tmp/cache",
    "LOGFIRE_TOKEN": "pylf_v1_eu_Z7cFS3v1bvD6mtHQ7rblY49FsBq5gKPPlNBQcZ7QZ7pp",
}

In [24]:
job = RayJob(
    job_name="rag-ingest",
    cluster_name=EXISTING_CLUSTER_NAME,
    namespace=NAMESPACE,
    entrypoint="python docling_milvus_process.py",
    runtime_env={
        "working_dir": ".",
        "excludes": [
            ".venv/",
            "__pycache__/",
            ".env",
            "*.egg-info/",
            "uv.lock",
            "manifests/",
            "docker/",
            "experiment_results.*",
            "*.ipynb",
            "*.md",
            "assets/", 
            ".logfire/",    
        ],
        "pip": pip_deps,
        "env_vars": pipeline_env,
    },
    active_deadline_seconds=7200,
)

job.submit()
print(f"RayJob '{job.name}' submitted.")

2026-04-06 13:10:17,312 Using existing cluster: raytest
2026-04-06 13:10:17,315 Initialized RayJob: rag-ingest in namespace: rag-example
2026-04-06 13:10:17,317 Zipping local working_dir: .
2026-04-06 13:10:17,320 Successfully zipped directory: . (16831 bytes) - Excluded 6 file(s) (.ipynb, .md)
2026-04-06 13:10:17,321 Zipping local working_dir: .
2026-04-06 13:10:17,323 Successfully zipped directory: . (16831 bytes) - Excluded 6 file(s) (.ipynb, .md)
2026-04-06 13:10:17,324 Added 21 environment variables to runtime_env
2026-04-06 13:10:17,324 Using pip dependencies from dict: 5 packages
2026-04-06 13:10:17,324 Local working_dir will be zipped, mounted, and unzipped to: /tmp/rayjob-working-dir
2026-04-06 13:10:17,326 Added init container to unzip working_dir to /tmp/rayjob-working-dir
2026-04-06 13:10:17,326 Built submitterPodTemplate with 1 files mounted at /home/ray/files, using image: quay.io/modh/ray@sha256:7b9c6d524b64a07746caa7dc89e691fc40eb4c2b4e41ffde8361bcd8d3c94d68
2026-04-06 

RayJob 'rag-ingest' submitted.


---
## Step 7 -- Monitor the Job

Track progress via the CodeFlare SDK, pod status, or job logs.


In [25]:
job.status()

          📦 CodeFlare RayJob Status 📦         
                                                
 ╭────────────────────────────────────────────╮ 
 │  Name                                      │ 
 │  rag-ingest                     Running ⚙  │ 
 │                                            │ 
 │  Job ID: rag-ingest-m2btz                  │ 
 │  Status: Running                           │ 
 │  RayCluster: raytest                       │ 
 │  Namespace: rag-example                    │ 
 │                                            │ 
 │  Started: 2026-04-06T13:10:17Z             │ 
 ╰────────────────────────────────────────────╯

(<CodeflareRayJobStatus.RUNNING: 2>, False)

In [ ]:
# List Ray pods
!oc get pods -n {NAMESPACE} -l ray.io/cluster={EXISTING_CLUSTER_NAME}

In [ ]:
# Fetch job logs (run after the job completes for the performance report)
!oc logs -l job-name={job.name} -n {NAMESPACE} --tail=100

---
## Step 9 -- Cleanup

Delete the RayJob to free resources. The RayCluster is managed separately.


In [ ]:
# Uncomment to delete the RayJob
# job.delete()
